<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller: Reducir 30 Mediciones a su Esencia con PCA, t-SNE e Isomap 🔬🗺️
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 05
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/homeworks/05_Reduccion_Dimensionalidad_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 05 (Reducción de Dimensionalidad)** con dos problemas nuevos: un laboratorio con **30 mediciones por biopsia** (¿cuántas dimensiones hacen falta de verdad?) y un sensor cuyas 3 lecturas dependen en realidad de **un solo parámetro oculto** enrollado en el espacio. Al terminar serás capaz de:

1. **Aplicar PCA** con criterio: escalar, elegir el número de componentes por varianza acumulada, medir el error de reconstrucción (y relacionarlo con los valores propios) y comprobar que se puede comprimir sin perder capacidad predictiva.
2. **Implementar PCA desde cero** con la SVD y verificarlo contra `scikit-learn`.
3. **Usar t-SNE** con distintas perplejidades, evaluar el resultado con *trustworthiness* y reconocer sus límites (no tiene `transform`).
4. **Aplicar Isomap** a una variedad enrollada, compararlo con PCA frente a una verdad terreno y diagnosticar **grafos de vecinos desconectados**.

> 📍 **Antes de empezar:** repasa [PCA](../05%20-%20Reduccion%20de%20Dimensionalidad/00_PCA.ipynb), [t-SNE](../05%20-%20Reduccion%20de%20Dimensionalidad/01_tSNE.ipynb) e [Isomap y variedades](../05%20-%20Reduccion%20de%20Dimensionalidad/02_Isomap_y_Variedades.ipynb). Si necesitas más guía, existe la [versión Para Dummies](Para%20Dummies/05_Reduccion_Dimensionalidad_Hands_On_Dummies.ipynb) de este taller.

---
## Instrucciones

1. **Resuelve de forma autónoma:** completa cada celda de código marcada con `# TODO:` respetando los **nombres de variables** que se piden (las autoverificaciones los usan).
2. Después de cada reto hay una celda de **autoverificación** con `assert`. Mientras el reto no esté resuelto **fallará con un mensaje** que te orienta; cuando todo esté bien imprimirá un ✅.
3. No borres ni modifiques las celdas de autoverificación ni la celda de preparación.
4. Antes de entregar ejecuta **Kernel → Restart & Run All** y comprueba que no queda ningún error ni ningún `# TODO` sin resolver.
5. Las respuestas escritas (conclusiones) van en **español** y deben apoyarse en tus números.

---
## Preparación del entorno y de los datos ⚙️

Ejecuta la celda siguiente (**no la modifiques**). No descarga nada: usa un *dataset* de `scikit-learn` y datos sintéticos con semilla fija.

- `X` / `y`: *Breast Cancer Wisconsin* (569 biopsias, 30 mediciones de los núcleos celulares; `y` = benigno/maligno). Se usa en los Retos 1 a 3.
- `idx_tsne`: submuestra de 300 biopsias para t-SNE (Reto 3), que es costoso.
- `Xr` / `t`: **rollo suizo** de 1000 puntos en 3D cuyo parámetro oculto `t` (la posición a lo largo de la espiral) es la **verdad terreno** (Reto 4).
- `rho_max(Y, t)`: ayudante que mide qué tan bien algún eje de la representación `Y` ordena los puntos como `t` (|correlación de Spearman|, máximo sobre los ejes).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, make_swiss_roll
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.manifold import TSNE, Isomap, trustworthiness
from sklearn.neighbors import kneighbors_graph
from scipy.sparse.csgraph import connected_components
from scipy.stats import spearmanr

SEED = 42
rng = np.random.default_rng(SEED)


def listo(*valores):
    """True si ninguna variable sigue sin resolver (None o ...)."""
    return all(v is not None and v is not Ellipsis for v in valores)


def rho_max(Y, t):
    """|Spearman| maximo entre alguna columna de Y y el parametro oculto t."""
    return max(abs(spearmanr(Y[:, j], t)[0]) for j in range(Y.shape[1]))


# --- Retos 1 a 3: biopsias ---------------------------------------------------------
datos = load_breast_cancer()
X, y = datos.data, datos.target
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
idx_tsne = rng.choice(len(X), 300, replace=False)

# --- Reto 4: rollo suizo con verdad terreno ----------------------------------------
Xr, t = make_swiss_roll(n_samples=1000, noise=0.05, random_state=SEED)

print("Biopsias:", X.shape, "| clases:", np.bincount(y), "| rollo suizo:", Xr.shape)
display(pd.DataFrame(X[:, :4], columns=datos.feature_names[:4]).describe().round(2).loc[["mean", "std"]])

---
## Reto 1: PCA con criterio — escala, varianza y reconstrucción 🎯 (25 puntos)

**Contexto.** El laboratorio sospecha que las 30 mediciones son muy redundantes. Debes cuantificar cuántas dimensiones bastan y comprobar que comprimir no daña un clasificador.

**Enunciado**

1. Estandariza `X` y guárdala en `Xs`. Ajusta `pca_full = PCA()` sobre `Xs`, guarda en `var_acum` la **varianza explicada acumulada** y en `n95` el **menor número de componentes** que alcanza al menos el 95 %.
2. Ajusta un `PCA()` sobre `X` **sin escalar** y guarda en `pc1_sin_escala` la fracción de varianza que captura su primer componente. Observa qué variable domina.
3. Reconstruye `Xs` con `n95` componentes (`transform` → `inverse_transform`). Guarda en `mse_rec` el **error cuadrático medio por elemento** (promedio de $(x-\hat x)^2$ sobre todas las entradas) y en `mse_formula` el valor que predice la teoría: la suma de los valores propios descartados, $\sum_{j>m}\lambda_j$, multiplicada por $\frac{n-1}{n\,d}$ (con $n$ filas y $d$ columnas).
4. Con `cross_val_score` y la validación `cv`, calcula la exactitud media de `make_pipeline(StandardScaler(), PCA(n95), LogisticRegression(max_iter=1000))` en `acc_pca` y la del mismo `Pipeline` **sin PCA** en `acc_base`. Usa `X` (el escalado va **dentro** del *pipeline* para no filtrar información).

**Criterios de aceptación**

- `n95` es coherente con `var_acum`; sin escalar, el primer componente concentra más del 95 % (la variable de mayor escala domina).
- `mse_rec` coincide con `mse_formula` (error relativo $<10^{-6}$).
- Con `n95` componentes la exactitud se mantiene: `acc_pca > 0.95` y no cae más de 0.02 frente a `acc_base`.

In [ ]:
# TODO 1: estandariza, ajusta PCA completo, varianza acumulada y n95
Xs = None
pca_full = None
var_acum = None
n95 = None

# TODO 2: fraccion de varianza del primer componente SIN escalar
pc1_sin_escala = None

# TODO 3: reconstruccion con n95 componentes y su error teorico
mse_rec = None
mse_formula = None

# TODO 4: exactitud en validacion cruzada con y sin PCA (el escalado va DENTRO del pipeline)
acc_pca = None
acc_base = None

In [ ]:
# ---- Autoverificacion del Reto 1 (no modificar) ----
assert listo(Xs, pca_full, var_acum, n95), "Reto 1: faltan Xs, pca_full, var_acum o n95 (paso 1)."
assert np.asarray(Xs).shape == X.shape and np.allclose(np.asarray(Xs).mean(0), 0, atol=1e-6) and np.allclose(np.asarray(Xs).std(0), 1, atol=1e-6), "Reto 1: Xs debe ser X estandarizada (media 0 y desviacion 1 por columna)."
assert len(var_acum) == 30 and np.all(np.diff(var_acum) >= 0) and abs(var_acum[-1] - 1) < 1e-9, "Reto 1: var_acum debe tener 30 valores crecientes que terminan en 1 (usa np.cumsum de explained_variance_ratio_)."
assert var_acum[n95 - 1] >= 0.95 and (n95 == 1 or var_acum[n95 - 2] < 0.95), f"Reto 1: n95 = {n95} no es el MENOR numero de componentes con >= 95 % de varianza."
assert listo(pc1_sin_escala) and pc1_sin_escala > 0.95, "Reto 1: sin escalar, el primer componente deberia capturar mas del 95 % de la varianza (paso 2)."
assert listo(mse_rec, mse_formula), "Reto 1: faltan mse_rec o mse_formula (paso 3)."
assert abs(mse_rec - mse_formula) / mse_formula < 1e-6, f"Reto 1: mse_rec ({mse_rec:.5f}) no coincide con la teoria ({mse_formula:.5f}). Revisa el promedio sobre TODOS los elementos y el factor (n-1)/(n*d)."
_n, _d = Xs.shape
assert abs(mse_formula - pca_full.explained_variance_[n95:].sum() * (_n - 1) / (_n * _d)) < 1e-9, "Reto 1: mse_formula debe sumar los valores propios DESCARTADOS (indices desde n95) de pca_full.explained_variance_."
assert listo(acc_pca, acc_base), "Reto 1: faltan acc_pca o acc_base (paso 4)."
assert acc_pca > 0.95, f"Reto 1: la exactitud con PCA deberia superar 0.95 y fue {acc_pca:.3f}. Recuerda escalar dentro del pipeline."
assert acc_pca >= acc_base - 0.02, f"Reto 1: con PCA la exactitud ({acc_pca:.3f}) no deberia caer mas de 0.02 frente a la base ({acc_base:.3f})."
_ref = cross_val_score(make_pipeline(StandardScaler(), PCA(n95), LogisticRegression(max_iter=1000)), X, y, cv=cv).mean()
assert abs(acc_pca - _ref) < 1e-6, "Reto 1: acc_pca no coincide con el pipeline pedido (StandardScaler -> PCA(n95) -> LogisticRegression) evaluado con cv."
print(f"✅ Reto 1 correcto | n95 = {n95} de 30 | PC1 sin escalar = {pc1_sin_escala:.3f} | exactitud {acc_pca:.3f} (PCA) vs {acc_base:.3f} (base)")

---
## Reto 2: PCA desde cero con la SVD 🧮 (20 puntos)

**Contexto.** Antes de confiar en una biblioteca conviene poder reproducirla. Si $X_c=U\Sigma V^\top$ es la SVD económica de los datos **centrados**, entonces:

- los componentes principales son las **filas de $V^\top$** (las primeras $m$),
- la varianza explicada de cada componente es $\lambda_j=\sigma_j^2/(n-1)$,
- los puntajes (proyección) son $Z=X_c V_m$.

**Enunciado.** Implementa `pca_svd(X, m)` que devuelva la tupla `(componentes, var_expl, Z)` con formas `(m, d)`, `(m,)` y `(n, m)`.

**Criterios de aceptación**

- Coincide con `PCA(m, svd_solver="full")` de `scikit-learn` (componentes y puntajes **salvo el signo**, y varianza explicada) en `Xs` y en un segundo conjunto con medias grandes (para que se note si no centras).
- Las filas de `componentes` son ortonormales y `var_expl` está ordenada de mayor a menor.
- La función **no modifica** el arreglo de entrada.

In [ ]:
def pca_svd(X, m):
    """PCA por SVD. Devuelve (componentes, var_expl, Z)."""
    # TODO: 1) centra X (sin modificar el original)  2) np.linalg.svd(..., full_matrices=False)
    #       3) componentes = primeras m filas de Vt   4) var_expl = s[:m]**2 / (n - 1)
    #       5) Z = Xc @ componentes.T
    pass

In [ ]:
# ---- Autoverificacion del Reto 2 (no modificar) ----
_Xs = StandardScaler().fit_transform(X)
_rg = np.random.default_rng(0)
_Xd = _rg.normal(size=(80, 6)) * np.array([5, 3, 2, 1, 0.5, 0.1]) + 10      # medias grandes
for _nombre, _D, _m in [("Xs", _Xs, 5), ("datos con media grande", _Xd, 3)]:
    _copia = _D.copy()
    _res = pca_svd(_D, _m)
    assert _res is not None and len(_res) == 3, "Reto 2: pca_svd debe devolver la tupla (componentes, var_expl, Z)."
    _C, _v, _Z = (np.asarray(a) for a in _res)
    assert np.array_equal(_D, _copia), "Reto 2: pca_svd no debe modificar la matriz de entrada (centra en una copia: Xc = X - media)."
    assert _C.shape == (_m, _D.shape[1]) and _v.shape == (_m,) and _Z.shape == (_D.shape[0], _m), f"Reto 2 ({_nombre}): formas esperadas (m, d), (m,) y (n, m); obtuviste {_C.shape}, {_v.shape}, {_Z.shape}."
    _ref = PCA(_m, svd_solver="full").fit(_D)
    _signo = np.sign((_C * _ref.components_).sum(axis=1))
    assert np.allclose(_C * _signo[:, None], _ref.components_, atol=1e-6), f"Reto 2 ({_nombre}): los componentes no coinciden con sklearn (salvo el signo). Usa las FILAS de Vt."
    assert np.allclose(_v, _ref.explained_variance_, rtol=1e-6), f"Reto 2 ({_nombre}): var_expl debe ser s**2 / (n - 1)."
    assert np.allclose(_Z * _signo, _ref.transform(_D), atol=1e-6), f"Reto 2 ({_nombre}): Z debe ser (X - media) @ componentes.T (revisa el centrado)."
    assert np.allclose(_C @ _C.T, np.eye(_m), atol=1e-8), "Reto 2: las filas de componentes deben ser ortonormales."
    assert np.all(np.diff(_v) <= 1e-12), "Reto 2: var_expl debe estar ordenada de mayor a menor."
print("✅ Reto 2 correcto | tu PCA por SVD coincide con scikit-learn (salvo el signo)")

---
## Reto 3: t-SNE, perplejidad y sus límites 🗺️ (15 puntos)

**Contexto.** El equipo clínico quiere un mapa 2D de las biopsias para inspeccionarlas visualmente. t-SNE promete conservar las **vecindades**, pero depende de la perplejidad y no es una transformación reutilizable.

**Enunciado.** Con la submuestra `Xt = Xs[idx_tsne]` (usa tu `Xs` del Reto 1) y `PERPLEJIDADES = [5, 30, 100]`:

1. Para cada perplejidad ajusta `TSNE(n_components=2, perplexity=p, init="pca", random_state=SEED)` y guarda: el mapa en `embeds[p]`, la divergencia final en `kl[p]` (`kl_divergence_`) y la `trustworthiness(Xt, mapa, n_neighbors=10)` en `trust[p]`.
2. Guarda en `mejor_perp` la perplejidad de mayor *trustworthiness*.
3. Calcula `pca2_trust`: la *trustworthiness* (con $k=10$) de la proyección de `Xt` a 2D con `PCA(2)`.
4. Guarda en `tsne_tiene_transform` un booleano que responda con **código** (`hasattr`) si `TSNE` ofrece un método `transform` para proyectar puntos nuevos.

**Criterios de aceptación**

- Tres mapas de forma `(300, 2)`, `kl` positivos y `trust` coherentes con una recomputación independiente.
- `mejor_perp` es consistente con `trust` y t-SNE supera a PCA en 2D en *trustworthiness*.
- `tsne_tiene_transform` es `False`: no puedes proyectar biopsias nuevas sobre un mapa ya hecho.

In [ ]:
PERPLEJIDADES = [5, 30, 100]
Xt = None            # TODO: Xs[idx_tsne]

embeds, kl, trust = {}, {}, {}
# TODO 1: recorre PERPLEJIDADES, ajusta TSNE y llena embeds, kl y trust

mejor_perp = None            # TODO 2
pca2_trust = None            # TODO 3
tsne_tiene_transform = None  # TODO 4: hasattr(TSNE, "transform")

In [ ]:
# ---- Autoverificacion del Reto 3 (no modificar) ----
assert listo(Xt, mejor_perp, pca2_trust, tsne_tiene_transform), "Reto 3: faltan Xt, mejor_perp, pca2_trust o tsne_tiene_transform."
_Xt = StandardScaler().fit_transform(X)[idx_tsne]
assert np.allclose(Xt, _Xt), "Reto 3: Xt debe ser Xs[idx_tsne] con Xs estandarizada (Reto 1)."
assert sorted(embeds) == sorted(PERPLEJIDADES) == sorted(kl) == sorted(trust), "Reto 3: embeds, kl y trust deben tener como claves las 3 perplejidades."
for p in PERPLEJIDADES:
    assert np.asarray(embeds[p]).shape == (300, 2), f"Reto 3: el mapa de perplejidad {p} debe tener forma (300, 2)."
    assert np.isfinite(kl[p]) and kl[p] > 0, f"Reto 3: kl[{p}] debe ser positivo (atributo kl_divergence_ del modelo ajustado)."
    _tr = trustworthiness(_Xt, np.asarray(embeds[p]), n_neighbors=10)
    assert abs(trust[p] - _tr) < 1e-9, f"Reto 3: trust[{p}] no coincide con trustworthiness(Xt, mapa, n_neighbors=10) = {_tr:.4f}."
    assert 0.85 < trust[p] <= 1, f"Reto 3: trust[{p}] = {trust[p]:.3f}; con init='pca' se esperaba entre 0.85 y 1."
assert mejor_perp == max(trust, key=trust.get), "Reto 3: mejor_perp debe ser la perplejidad con mayor trustworthiness."
assert abs(pca2_trust - trustworthiness(_Xt, PCA(2).fit_transform(_Xt), n_neighbors=10)) < 1e-9, "Reto 3: pca2_trust debe usar PCA(2) sobre Xt y n_neighbors=10."
assert trust[mejor_perp] > pca2_trust, "Reto 3: el mejor t-SNE deberia superar a PCA en 2D en trustworthiness."
assert tsne_tiene_transform is False, "Reto 3: TSNE no tiene transform; calcula el booleano con hasattr(TSNE, 'transform')."
print(f"✅ Reto 3 correcto | trustworthiness: {({p: round(v, 3) for p, v in trust.items()})} | PCA 2D = {pca2_trust:.3f} | mejor perplejidad = {mejor_perp}")

---
## Reto 4: Isomap frente a PCA en una variedad enrollada 🌀 (25 puntos)

**Contexto.** Un sensor entrega 3 lecturas `Xr` que en realidad dependen de **un solo parámetro oculto** `t` (por ejemplo, el desgaste acumulado de una pieza), pero las lecturas describen una superficie **enrollada** como un rollo. Queremos "desenrollarla" a 2D. Como conocemos `t`, podemos medir qué técnica lo recupera.

**Enunciado**

1. Calcula `Y_iso = Isomap(n_neighbors=10, n_components=2).fit_transform(Xr)` y `Y_pca = PCA(2).fit_transform(Xr)`.
2. Guarda `rho_iso = rho_max(Y_iso, t)` y `rho_pca = rho_max(Y_pca, t)` (qué tan bien algún eje ordena los puntos como `t`).
3. Guarda `trust_iso` y `trust_pca`: la `trustworthiness(Xr, Y, n_neighbors=10)` de cada mapa.
4. Diagnostica el **grafo de vecinos**: para cada $k\in\{1,2,3,5,10\}$ construye `kneighbors_graph(Xr, k)` y cuenta sus componentes conexas con `connected_components(..., directed=False)`; guarda el diccionario `n_comp = {k: número de componentes}` y `k_conexo`, el **menor** $k$ de la lista que deja el grafo conectado (1 componente).

**Criterios de aceptación**

- Isomap recupera el parámetro oculto ($\rho>0{,}8$) y PCA no ($\rho<0{,}4$).
- Las *trustworthiness* son coherentes con una recomputación independiente (ambas $>0{,}85$).
- Con pocos vecinos el grafo está fragmentado (`n_comp[1] > 1`), con $k=10$ está conectado, y el número de componentes nunca aumenta con $k$.

In [ ]:
# TODO 1: las dos proyecciones 2D
Y_iso = None
Y_pca = None

# TODO 2: recuperacion del parametro oculto
rho_iso = None
rho_pca = None

# TODO 3: trustworthiness (k = 10) de cada mapa
trust_iso = None
trust_pca = None

# TODO 4: componentes conexas del grafo de k vecinos
n_comp = {}      # {k: numero de componentes conexas}
k_conexo = None  # menor k de [1, 2, 3, 5, 10] con un solo componente

In [ ]:
# ---- Autoverificacion del Reto 4 (no modificar) ----
assert listo(Y_iso, Y_pca, rho_iso, rho_pca, trust_iso, trust_pca, k_conexo), "Reto 4: faltan variables por definir (revisa los 4 pasos)."
assert np.asarray(Y_iso).shape == (1000, 2) and np.asarray(Y_pca).shape == (1000, 2), "Reto 4: Y_iso y Y_pca deben tener forma (1000, 2)."
assert abs(rho_iso - rho_max(np.asarray(Y_iso), t)) < 1e-9 and abs(rho_pca - rho_max(np.asarray(Y_pca), t)) < 1e-9, "Reto 4: rho_iso y rho_pca deben calcularse con rho_max(Y, t) sobre TUS proyecciones."
assert rho_iso > 0.8, f"Reto 4: Isomap deberia recuperar t (rho > 0.8) y obtuvo {rho_iso:.3f}. Revisa n_neighbors=10 y que ajustes sobre Xr."
assert rho_pca < 0.4, f"Reto 4: PCA no deberia desenrollar el rollo (rho < 0.4) y obtuvo {rho_pca:.3f}."
for _nombre, _Y, _v in [("Isomap", Y_iso, trust_iso), ("PCA", Y_pca, trust_pca)]:
    _tr = trustworthiness(Xr, np.asarray(_Y), n_neighbors=10)
    assert abs(_v - _tr) < 1e-9, f"Reto 4: trust de {_nombre} no coincide con trustworthiness(Xr, Y, n_neighbors=10) = {_tr:.4f}."
    assert _v > 0.85, f"Reto 4: trust de {_nombre} = {_v:.3f}; se esperaba > 0.85."
assert sorted(n_comp) == [1, 2, 3, 5, 10], "Reto 4: n_comp debe tener como claves k = 1, 2, 3, 5, 10."
for _k, _c in n_comp.items():
    assert _c == connected_components(kneighbors_graph(Xr, _k), directed=False)[0], f"Reto 4: n_comp[{_k}] no coincide. Usa kneighbors_graph(Xr, {_k}) y connected_components(..., directed=False)."
assert n_comp[1] > 1, "Reto 4: con k = 1 el grafo debe estar fragmentado en muchas islas."
assert n_comp[10] == 1, "Reto 4: con k = 10 el grafo debe estar conectado (1 componente)."
_cs = [n_comp[k] for k in sorted(n_comp)]
assert all(a >= b for a, b in zip(_cs, _cs[1:])), "Reto 4: al aumentar k el numero de componentes no puede aumentar."
assert k_conexo == min(k for k, c in n_comp.items() if c == 1), "Reto 4: k_conexo es el MENOR k de la lista con un solo componente."
print(f"✅ Reto 4 correcto | rho Isomap = {rho_iso:.3f} vs PCA = {rho_pca:.3f} | componentes por k: {n_comp} | k_conexo = {k_conexo}")

---
## Reto 5: Conclusiones 🧠 (15 puntos)

*Sin código.* Escribe entre **5 y 8 líneas** que interpreten **tus** resultados (cita `n95`, `pc1_sin_escala`, `mse_rec`, `acc_pca`/`acc_base`, `trust`, `pca2_trust`, `rho_iso`/`rho_pca`, `n_comp`). Debes responder, como mínimo:

1. ¿Qué te dice `n95` sobre la redundancia de las 30 mediciones y por qué el PCA sin escalar es engañoso?
2. ¿Por qué la exactitud casi no cambió al comprimir? ¿Qué relación hay entre el error de reconstrucción y los valores propios descartados?
3. Si quisieras proyectar la biopsia de un paciente nuevo, ¿usarías PCA o t-SNE? ¿Por qué? ¿Qué harías con las distancias entre grupos de un mapa t-SNE?
4. ¿Por qué PCA no desenrolla el rollo y qué riesgo corre Isomap si eliges un $k$ demasiado pequeño o demasiado grande?

**Tus conclusiones (edita esta celda):**

> *(escribe aquí)*

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: PCA con escala, varianza acumulada, reconstrucción y *pipeline* | 25 |
| Reto 2: PCA desde cero con la SVD verificada contra `scikit-learn` | 20 |
| Reto 3: t-SNE con varias perplejidades, *trustworthiness* y límites | 15 |
| Reto 4: Isomap frente a PCA con verdad terreno y diagnóstico del grafo | 25 |
| Reto 5: conclusiones (5-8 líneas, con cifras propias y razonamiento correcto) | 15 |
| **Total** | **100** |

*Las autoverificaciones comprueban propiedades y rangos; el docente puede revisar además la calidad del código y de la interpretación.*

### ✅ Checklist de entrega

- [ ] Ejecuté **Restart & Run All** y no hay errores ni celdas `# TODO` sin resolver.
- [ ] Las cuatro autoverificaciones imprimen ✅.
- [ ] Respondí las cuatro preguntas de las conclusiones con cifras de mis propios resultados.
- [ ] No modifiqué las celdas de preparación ni de autoverificación.
- [ ] Entregué el notebook con los resultados visibles (salidas guardadas).

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Machine Learning</i>
  </p>
</div>